# 02. Merge Medicare, income, and rurality

Builds two tables in `medicare.duckdb`:

* **`rucc`**: USDA Rural-Urban Continuum Codes, one row per county
* **`merged`**: `analysis` + SAIPE income/poverty (by county and year) + RUCC (by county)

Needs `analysis` and `saipe`, which `01_build_database.ipynb` creates.

> DuckDB lets only one program write to the database file at a time. If another
> notebook has it open, run `con.close()` there first.

Both tables are rebuilt from source on every run (`CREATE OR REPLACE`), so rerunning this notebook is safe.

In [ ]:
import duckdb

con = duckdb.connect("../medicare.duckdb")

## 1. Pivot RUCC from long to wide

The raw file has three rows per county (population, RUCC code, description).
`GROUP BY FIPS` collapses them to one row; each `CASE WHEN` keeps only the value
for one attribute, and `MAX` picks that single non-null value.
This is the SQL version of Stata's `reshape wide` or R's `pivot_wider()`.

`all_varchar=true` reads everything as text so FIPS codes keep their leading zeros.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE rucc AS
SELECT
    FIPS AS fips,
    MAX(CASE WHEN Attribute = 'RUCC_2023'       THEN TRY_CAST(Value AS INTEGER) END) AS rucc,
    MAX(CASE WHEN Attribute = 'Population_2020' THEN TRY_CAST(Value AS INTEGER) END) AS pop_2020,
    MAX(CASE WHEN Attribute = 'Description'     THEN Value END)                     AS rucc_desc
FROM read_csv('../data/rucc_2023.csv', encoding='latin-1', all_varchar=true)
GROUP BY FIPS
""")

con.sql("SELECT COUNT(*) AS counties, COUNT(rucc) AS with_code FROM rucc")

## 2. Merge

* SAIPE joins on county **and** year.
* RUCC 2023 is a single snapshot, so it joins on county only.
* Shannon County, SD (46113) was renamed Oglala Lakota (46102) in 2015 with the
  same borders, so its 2014 row is matched to the new code.
* `metro_rural`: RUCC 1-3 = Metro, 4-9 = Rural (nonmetro).

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE merged AS
SELECT a.*,
       s.median_income, s.poverty_rate,
       r.rucc, r.pop_2020, r.rucc_desc,
       CASE WHEN r.rucc <= 3 THEN 'Metro'
            WHEN r.rucc >= 4 THEN 'Rural' END AS metro_rural
FROM analysis a
LEFT JOIN saipe s ON a.fips = s.fips AND a.year = s.year
LEFT JOIN rucc  r ON r.fips = CASE WHEN a.fips = '46113' THEN '46102' ELSE a.fips END
""")

## 3. Match rate

In [ ]:
con.sql("""
SELECT COUNT(*)                                           AS rows,
       ROUND(100.0 * COUNT(median_income) / COUNT(*), 2)  AS pct_with_saipe,
       ROUND(100.0 * COUNT(rucc) / COUNT(*), 2)           AS pct_with_rucc
FROM merged
""")

Which rows did not match, and why?

In [ ]:
con.sql("""
SELECT fips, county_name, MIN(year) AS first_year, MAX(year) AS last_year, COUNT(*) AS rows
FROM merged
WHERE rucc IS NULL OR median_income IS NULL
GROUP BY ALL
ORDER BY fips
""")

All unmatched rows are FIPS changes that RUCC 2023 does not cover:

* **Connecticut, 2014-2021** (8 old counties): CMS and SAIPE both switched to the
  9 planning regions in 2022, so they match each other every year, but RUCC 2023
  only has the planning regions. There is no one-to-one mapping from old counties to regions.
* **Valdez-Cordova, AK (02261), 2014-2018**: split into two census areas in 2019.

These rows keep their spending and income data but have no `metro_rural` value,
so they drop out of anything that uses rurality (about 0.2% of rows).

In [ ]:
con.sql("SELECT metro_rural, COUNT(*) AS rows FROM merged GROUP BY 1 ORDER BY 1")

In [ ]:
con.close()  # release the file so other notebooks can open it